# 在 Google Colab 上训练演示版「对照 AI」（约 30–60 分钟）

1. 菜单 **代码执行程序 → 更改运行时类型 → GPU（T4 即可）**
2. 依次运行下面的单元格。第一个单元格会让你上传 `groh2024_repro_kit.zip`。
3. 最后会下载 `checkpoint.zip`，解压到本地工具包的 `dls/checkpoint/` 目录，演示时即可用 `dls/predict.py` 推理。

说明：论文的对照 AI 还用了作者私有的教科书/图谱图片（共 31,219 张），这里只用公开的 Fitzpatrick17k，所以准确率不会与论文一致；它是讲解训练流程用的演示模型。

In [ ]:
from google.colab import files
up = files.upload()            # 选择 groh2024_repro_kit.zip
!unzip -q -o groh2024_repro_kit.zip
%cd groh2024_repro_kit
!nvidia-smi -L

In [ ]:
# Fitzpatrick17k 的标注表（如果链接失效，到 https://github.com/mattgroh/fitzpatrick17k 手动下载后上传）
!wget -q -O fitzpatrick17k.csv https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv
!head -3 fitzpatrick17k.csv
# 9 类（8 种主要疾病 + Other）每类最多 300 张；部分原始链接可能失效，会自动跳过
!python dls/fetch_f17k_images.py --csv fitzpatrick17k.csv --mode training --per-class 300 --out-dir f17k_images --train-csv f17k_train.csv

In [ ]:
!python dls/train_control_dls.py --csv f17k_train.csv --img-root f17k_images --epochs 6 --batch-size 64

In [ ]:
# Wizard of Oz：把验证集上的错误预测随机改对 65%，得到「增强 AI」
!python dls/wizard_of_oz.py --pred dls/checkpoint/val_predictions.csv --out dls/checkpoint/treatment_predictions.csv

In [ ]:
!zip -q -r checkpoint.zip dls/checkpoint
files.download('checkpoint.zip')